In [6]:
y = """
a1: &a1
    b: 1
    c:
        d: 2
        e: 3
a2: &a2
    <<: *a1
a3: &a3
    <<: *a2
c:
    <<: *a2
    h: 4
    MERGE: adfd
    c:
        f: 100
"""

import yaml

yaml.safe_load(y)

{'a1': {'b': 1, 'c': {'d': 2, 'e': 3}},
 'a2': {'b': 1, 'c': {'d': 2, 'e': 3}},
 'a3': {'b': 1, 'c': {'d': 2, 'e': 3}},
 'c': {'b': 1, 'c': {'f': 100}, 'h': 4, 'MERGE': 'adfd'}}

In [5]:
import yaml
from yaml.loader import SafeLoader
from yaml.nodes import MappingNode, SequenceNode

MERGE_KEY = "<<"


class DeepMergeLoader(SafeLoader):
    pass


def deep_merge(left, right):
    # Same policy as above (adjust to needs)
    if isinstance(left, dict) and isinstance(right, dict):
        out = dict(left)
        for k, rv in right.items():
            out[k] = deep_merge(out[k], rv) if k in out else rv
        return out
    elif isinstance(left, list) and isinstance(right, list):
        return left + right
    else:
        return right


# Adapted from PyYAML’s construct_mapping, with custom merge handling
def construct_mapping_with_deep_merge(loader, node, deep=False):
    if not isinstance(node, MappingNode):
        raise yaml.constructor.ConstructorError(
            None, None, f"expected a mapping node, but found {node.id}", node.start_mark
        )

    loader.flatten_mapping(node)  # resolves aliases to nodes; keeps << entries
    mapping = {}
    merges = []

    for key_node, value_node in node.value:
        key = loader.construct_object(key_node, deep=deep)
        if key == MERGE_KEY:
            # Value can be a mapping or a sequence of mappings
            if isinstance(value_node, MappingNode):
                merges.append(loader.construct_mapping(value_node, deep=deep))
            elif isinstance(value_node, SequenceNode):
                for subnode in value_node.value:
                    if not isinstance(subnode, MappingNode):
                        raise yaml.constructor.ConstructorError(
                            "while constructing a mapping",
                            node.start_mark,
                            "expected a mapping for merging, but found %s" % subnode.id,
                            subnode.start_mark,
                        )
                    merges.append(loader.construct_mapping(subnode, deep=deep))
            else:
                raise yaml.constructor.ConstructorError(
                    "while constructing a mapping",
                    node.start_mark,
                    "expected a mapping or list of mappings for merging, but found %s" % value_node.id,
                    value_node.start_mark,
                )
        else:
            value = loader.construct_object(value_node, deep=deep)
            mapping[key] = value

    # Apply merges left-to-right with deep policy, then overlay own keys
    merged = {}
    for m in merges:
        merged = deep_merge(merged, m)
    result = deep_merge(merged, mapping)
    return result


DeepMergeLoader.add_constructor(yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, construct_mapping_with_deep_merge)


def safe_load_deep(stream):
    return yaml.load(stream, Loader=DeepMergeLoader)


yaml_str = """
a2:
    MERGE: aa
    c:
        e: 5
    MERGE: bb
"""
safe_load_deep(yaml_str)


{'a2': {'MERGE': 'bb', 'c': {'e': 5}}}

In [8]:
yaml_str = """
a2:
    **: aa
    c:
        e: 5
    MERGE: bb
"""
yaml.safe_load(yaml_str)

ScannerError: while scanning an alias
  in "<unicode string>", line 3, column 5:
        **: aa
        ^
expected alphabetic or numeric character, but found '*'
  in "<unicode string>", line 3, column 6:
        **: aa
         ^

In [7]:
class Hero:
    def __init__(self, name, hp, sp):
        self.name = name
        self.hp = hp
        self.sp = sp
    def __repr__(self):
        return "%s(name=%r, hp=%r, sp=%r)" % (
            self.__class__.__name__, self.name, self.hp, self.sp)
    
yaml.safe_load("""
!!python/object:__main__.Hero
name: Welthyr Syxgon
hp: 1200
sp: 0
""")

ConstructorError: could not determine a constructor for the tag 'tag:yaml.org,2002:python/object:__main__.Hero'
  in "<unicode string>", line 2, column 1:
    !!python/object:__main__.Hero
    ^